# HPO Integration Tests

Automated smoke tests for the SolarSystemLander HPO production path. These tests check Python objects and integrations, not model quality.

## Setup

In [ ]:
# cell: setup

from pathlib import Path
import os
import subprocess
import sys

if not Path("hpo/src").exists():
    if not Path("/content").exists():
        raise RuntimeError("Run this notebook from the rl_lab repository root.")
    subprocess.run(["git", "clone", "https://github.com/miwehle/rl_lab.git"], check=True)
    os.chdir("rl_lab")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "hpo/requirements.txt"],
    check=True,
)

for _path in ("dqn/src", "hpo/src"):
    if _path not in sys.path:
        sys.path.insert(0, _path)

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
# cell: check-helpers; requires: setup

CHECK_RESULTS = []

def record_pass(name, **details):
    CHECK_RESULTS.append({"check": name, "status": "PASS"} | details)
    print(f"PASS {name}")

## Check 1: Environment Factory Smoke

In [ ]:
# cell: environment-factory-smoke; requires: check-helpers

from hpo.solar_system_lander.environment import EnvFactory, World

_factory = EnvFactory("10d", world_mix={World.MOON: 1, World.MARS: 1})
_eval_envs = _factory.evaluation_envs()
assert list(_eval_envs) == ["moon", "mars"]

_env = _eval_envs["moon"]()
try:
    _obs, _ = _env.reset(seed=123)
    assert _obs.shape == _env.observation_space.shape
    _obs, _reward, _terminated, _truncated, _ = _env.step(_env.action_space.sample())
    assert _obs.shape == _env.observation_space.shape
finally:
    _env.close()

_vector_env = _factory.make_training_env(num_envs=2)
try:
    _obs, _ = _vector_env.reset(seed=123)
    assert _obs.shape[0] == 2
    _obs, _rewards, _terminated, _truncated, _ = _vector_env.step(_vector_env.action_space.sample())
    assert _obs.shape[0] == 2
finally:
    _vector_env.close()

record_pass("Environment Factory Smoke", worlds=", ".join(_eval_envs))

## Check 2: Objective Smoke

In [ ]:
# cell: objective-smoke; requires: check-helpers

from hpo.hyperparams import HP
from hpo.objective import ObjectiveConfig, create_objective
from hpo.solar_system_lander.environment import EnvFactory, World

class _FixedTrial:
    number = 0

    def __init__(self):
        self.params = {}
        self.user_attrs = {}

    def set_user_attr(self, name, value):
        self.user_attrs[name] = value

_params = {
    HP.LEARNING_RATE: 0.001,
    HP.BATCH_SIZE: 8,
    HP.EPS_END: 0.1,
    HP.EPS_DECAY: 100,
    HP.GAMMA: 0.99,
    HP.TAU: 0.005,
    HP.LEARNING_STARTS: 4,
    HP.OPTIMIZE_EVERY: 1,
    HP.REPLAY_MEMORY: 512,
    HP.NUM_EPISODES: 2,
    HP.HIDDEN_SIZE: 32,
}
_objective_cfg = ObjectiveConfig(
    environment_factory=EnvFactory("10d", world_mix={World.MOON: 1}),
    num_envs=1,
    device=device,
    eval_episodes=1,
    eval_max_steps=100,
    eval_seed=10_000,
    training_seed=42,
    early_stopping_score=None,
)
_objective = create_objective(
    suggest_parameter_values=lambda _trial, _incumbent_params: None,
    incumbent_params=_params,
    config=_objective_cfg,
)
_trial = _FixedTrial()
_score = _objective(_trial)

assert isinstance(_score, float)
assert _trial.user_attrs["trained_episodes"] >= _params[HP.NUM_EPISODES]
assert _trial.user_attrs["env_steps"] > 0
assert _trial.user_attrs["optimizer_updates"] >= 0
assert len(_trial.user_attrs["training_curve"]["episode_returns"]) >= _params[HP.NUM_EPISODES]

record_pass(
    "Objective Smoke",
    score=round(_score, 1),
    trained_episodes=_trial.user_attrs["trained_episodes"],
)

## Check 3: Checkpoint Roundtrip

In [ ]:
# cell: checkpoint-roundtrip; requires: check-helpers

import math
import tempfile

import torch

from dqn.model import DQN
from hpo.checkpointing import load_checkpoint, save_checkpoint
from hpo.objective import evaluate_greedy_q_net
from hpo.solar_system_lander.environment import EnvFactory, World

_factory = EnvFactory("10d", world_mix={World.MOON: 1})
_make_env = _factory.evaluation_envs()["moon"]
_env = _make_env()
try:
    _obs, _ = _env.reset(seed=123)
    _n_observations = math.prod(tuple(_obs.shape))
    _n_actions = _env.action_space.n
finally:
    _env.close()

_q_net = DQN(_n_observations, _n_actions, hidden_size=32).to(device)
with tempfile.TemporaryDirectory() as _tmp_dir:
    _checkpoint_path = Path(_tmp_dir) / "roundtrip.pt"
    save_checkpoint(_q_net, _checkpoint_path, {"score": 1.0, "source": "integration-test"})

    _loaded = DQN(_n_observations, _n_actions, hidden_size=32).to(device)
    _metadata = load_checkpoint(_loaded, _checkpoint_path, device)

    for _left, _right in zip(_q_net.parameters(), _loaded.parameters()):
        assert torch.equal(_left, _right)

    _score = evaluate_greedy_q_net(
        q_net=_loaded,
        device=device,
        make_env=_make_env,
        episodes=1,
        max_steps=100,
        seed=10_000,
    )

assert _metadata["score"] == 1.0
assert isinstance(_score, float)
record_pass("Checkpoint Roundtrip", eval_score=round(_score, 1))

## Summary

In [ ]:
# cell: summary; requires: environment-factory-smoke, objective-smoke, checkpoint-roundtrip

import pandas as pd

display(pd.DataFrame(CHECK_RESULTS))